# tl;dr

PN16 tests the ordered whole-wave rule `A+B -> AB`, `AB+BA -> next rung` on exact prime-wheel sieving.

The result is precise: forward `AB` and reverse `BA` have different partial histories, but their completed masks are
identical. Recombining that completed identity with its reversal is idempotent and does **not** make the next rung.
The completed p17 web instead locates `19` as its first quiet node; retaining 19 as the new relation removes exactly
one lift of every p17 survivor and constructs the p19 wheel exactly. Independent validation passed 71/71 checks.


In [1]:
from pathlib import Path
import csv, hashlib, json, math

HERE = Path(r'F:\SystemFormulaFolder\GIT\ARA-GIT\analysis\primes')
def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest().upper()

result = json.loads((HERE / 'PN16_ORDERED_WHOLE_WAVE_LIFT_RESULTS.json').read_text(encoding='utf-8'))
validation = json.loads((HERE / 'PN16_ORDERED_WHOLE_WAVE_LIFT_VALIDATION.json').read_text(encoding='utf-8'))
print(result['status'])
print('Protocol hash:', result['protocol_sha256'])
print('Independent validation:', validation['passed_count'], '/', validation['check_count'])


ORDERED PATH RETAINED / COMPLETED AB=BA / REVERSED COPY IDEMPOTENT / QUIET-NODE RELATION LIFTS NEXT RUNG
Protocol hash: 281DAE4D278A6781D9CD42D0D07F7CF36E99DE397126137773719C2B5902373F
Independent validation: 71 / 71


# Context & Methods

For the first `k` prime gates, `AB` applies the gates in ascending order and `BA` applies the same gates in
descending order. Each intermediate survivor mask is retained. The completed masks are then compared with the
direct coprimality identity.

For the vertical lift, the first integer above the terminal prime that survives the complete parent web is recovered
without supplying the next prime to the builder. That quiet node becomes the new gate.

This is an exact deterministic structural test. It is not statistical estimation and has no sampling uncertainty.
It is also not historically blind: code isolation checks the translation and implementation, not the established
prime sequence's novelty.


In [2]:
print('terminal | period | phi(parent) | max AB/BA partial disagreement | completed disagreement | quiet node')
for row in result['materialized_rungs']:
    print(f"{row['terminal_prime']:>8} | {row['period']:>7,} | {row['expected_totient']:>11,} | "
          f"{row['max_partial_hamming_fraction']:.6f} | {row['final_hamming_count']:>22} | "
          f"{row['first_quiet_node']}")


terminal | period | phi(parent) | max AB/BA partial disagreement | completed disagreement | quiet node
       5 |      30 |           8 | 0.500000 |                      0 | 7
       7 |     210 |          48 | 0.561905 |                      0 | 11
      11 |   2,310 |         480 | 0.593074 |                      0 | 13
      13 |  30,030 |       5,760 | 0.613054 |                      0 | 17
      17 | 510,510 |      92,160 | 0.635239 |                      0 | 19


# Data

The data are exact integer survivor masks over complete primorial periods. Development parents end at
`5, 7, 11, 13`; the code-isolated target parent ends at `17`. The target lift has period
`17# * 19 = 9,699,690`. A separate lightweight check repeats the quiet-node identity for every consecutive prime
pair through terminal prime 997.


In [3]:
paths = list(csv.DictReader((HERE / 'PN16_ORDERED_WHOLE_WAVE_LIFT_PATHS.csv').open(encoding='utf-8')))
target_paths = [row for row in paths if int(row['terminal_prime']) == 17]
print('p17 ordered path')
print('depth | forward gate | reverse gate | forward survivors | reverse survivors | disagreement')
for row in target_paths:
    print(f"{int(row['depth']):>5} | {int(row['forward_gate']):>12} | {int(row['reverse_gate']):>12} | "
          f"{int(row['forward_survivors']):>17,} | {int(row['reverse_survivors']):>17,} | "
          f"{float(row['hamming_fraction']):.6f}")


p17 ordered path
depth | forward gate | reverse gate | forward survivors | reverse survivors | disagreement
    1 |            2 |           17 |           255,255 |           480,480 | 0.500000
    2 |            3 |           13 |           170,170 |           443,520 | 0.622926
    3 |            5 |           11 |           136,136 |           403,200 | 0.635239
    4 |            7 |            7 |           116,688 |           345,600 | 0.544491
    5 |           11 |            5 |           106,080 |           276,480 | 0.388318
    6 |           13 |            3 |            97,920 |           184,320 | 0.191808
    7 |           17 |            2 |            92,160 |            92,160 | 0.000000


# Results

The decisive distinction is between a **path relation** and a **completed identity**. Order is strongly visible
during the path, yet the final projection commutes. The next rung is created only after the parent web identifies
the new quiet node and that new gate is retained.


In [4]:
lift = result['target_lift']
print('Recovered quiet node:', lift['recovered_quiet_node'])
print('Parent period:', f"{lift['parent_period']:,}")
print('Child period:', f"{lift['child_period']:,}")
print('Parent survivors repeated 19 times:', f"{lift['tiled_parent_survivors']:,}")
print('Newly released by gate 19:', f"{lift['newly_released']:,}")
print('Child survivors:', f"{lift['child_survivors']:,}")
print('Missing relation among parent survivors:', f"{lift['missing_relation_fraction_given_parent_survival']:.9f}")
print('Same identity + reversal equals child:', lift['same_identity_recombination_equals_child'])
print('New gate lift equals direct child:', lift['lifted_equals_direct_child'])
print('Quiet-node theorem-scale checks:', result['theorem_scale_quiet_nodes']['pair_count'], 'all pass =', result['theorem_scale_quiet_nodes']['all_pass'])


Recovered quiet node: 19
Parent period: 510,510
Child period: 9,699,690
Parent survivors repeated 19 times: 1,751,040
Newly released by gate 19: 92,160
Child survivors: 1,658,880
Missing relation among parent survivors: 0.052631579
Same identity + reversal equals child: False
New gate lift equals direct child: True
Quiet-node theorem-scale checks: 168 all pass = True


# Takeaways

1. The user's ordered-coupling intuition survives: `AB` and `BA` are distinguishable while the process is open.
2. A completed sieve whole and its simple reversal are not independent next-rung poles; they coarse-grain to one
   identical parent identity.
3. The current parent nevertheless contains an exact bottom-up next-prime rule: its first quiet node is the next
   prime.
4. The most faithful Information³ reading at this grain is therefore **parent whole + next survivor + their new gate
   relation**, not **parent whole + reversed copy alone**.
5. This is an exact ARA crosswalk of the recursive wheel sieve, not a faster prime algorithm or a new prime theorem.
